# 132588 — sparse-grid point generation to GENE parameters files

**What this does.** Seed 132588 profiles + EQDSK → sparse-grid sampler proposes
points in three scaling axes → each point is reconstructed through CHEASE-BS into
its *own* EQDSK (filename carries the transform) and its *own* iterdb → each pair
is written into its *own* GENE parameters file under a temp directory.

**What this does not do.** Nothing is submitted. No GENE runs. No growth rates.
`_default_submitter` and `_job_finished` stay unexercised — the only never-run
campaign function this touches is `_write_parameters`.

**Read this before running.** With no GENE runs there is no QoI, and the grid
will not produce a second batch until the first is told. To get more than the
single seed point out of the sampler, this notebook tells it **fabricated**
values (`dummy_qoi`). The `session.json` it leaves behind is meaningless as a
scan and must never be resumed by a real campaign. Set `TARGET_POINTS = 1` if
you want only the single honest seed point.

Run on NERSC — CHEASE-BS needs the compiled binary via
`TPED/config/user_config.yaml`.

## 1. Inputs you have to provide

These are NERSC-side files. Fill in every one that is `None`, then run the check
cell below and do not continue until it prints `all inputs present`.

In [ ]:
# ---------------------------------------------------------------- REQUIRED

# Baseline GENE parameters file this scan perturbs. Neither this repo nor TPED
# has one for 132588 — high_triangularity/132588 holds only analysis notebooks.
# The 129015 one is the shape assumed here (magn_geometry='tracer_efit', EQDSK
# geomfile, profiles via iterdb_file, x0 set to the analysis radius):
#   TPED/data/discharges/NSTX129015/r_0.85_NE/scanfiles0000/parameters
BASE_PARAMETERS = None

# The seed discharge. Two ways in — pick ONE and leave the other None.
#
#   SEED_DIRPATH  : directory holding 132588's EQDSK + pfile (or GENE profiles).
#                   Auto-discovered, same as the CHEASE-BS canary. PREFERRED:
#                   a pfile carries both rho_tor and rho_pol.
#   SEED_ITERDB   : a 132588 iterdb, converted to GENE profiles files on the way
#                   in. Needs SEED_GFILE alongside it, since CHEASE-BS
#                   reconstructs *from* a source equilibrium and an iterdb has
#                   no geometry. Caveat: an iterdb has rho_tor only, so the
#                   rho_pol column of the converted profiles is filled with
#                   rho_tor.
SEED_DIRPATH = "/global/homes/j/joeschm/data/ST_research/NSTXU_discharges/132588"
SEED_ITERDB  = None
SEED_GFILE   = None            # required only when using SEED_ITERDB

# ---------------------------------------------------------------- OPTIONAL

# Where the generated equilibria and parameters files land. Gitignored.
OUTROOT = "tmp_runs"

# Single k_y for the inner scanlist. The QoI reduction names the same value,
# though nothing is harvested here.
KY = 0.05

# How many points to generate, as a BUDGET rather than an exact count.
#
# An adaptive sparse grid picks its own batch sizes: batch 0 is a single seed
# node, and every later batch's size depends on which axes the previous batch's
# values made look important. So "give me exactly 12 points" is not a request
# the algorithm can honour. Refinement continues until at least TARGET_POINTS
# evaluations have been proposed, then stops at the end of that batch -- so the
# real total lands at or just above the target, overshooting by at most one
# batch. Every point is a CHEASE-BS run, so this is the cost knob.
# 5 points on three axes is the seed node plus four one-axis moves -- each curve
# differs from the centre in exactly one way, which is what makes the profile
# plot readable. Raising this adds points that move two axes at once.
TARGET_POINTS = 5

# The scan box, as SCALE FACTORS on the nominal mtanh fit of whichever discharge
# is seeded. Set to None to use TPED's SCAN_BOUNDS instead.
#
# Widened from TPED's +/-30% because this run is a pipeline test whose output is
# read off a plot: the point of it is to see the pedestal move. +/-40% on the
# pedestal height and 0.6-1.4 on the width put visible daylight between the
# curves. That makes the box wider than the Hatch grid precedent the physics
# bounds are drawn from, so it is the wrong box for a real scan -- narrow it
# back to PILOT_BOUNDS before any result is kept.
#
# Also note the gate can reject a point at the edges, and a rejected point ends
# the run (the grid cannot advance past a node with no value). If the run stops
# on "batch incomplete", pull these in rather than assuming the code broke.
SCAN_BOUNDS = {
    "Te_ped_scale":   (0.6, 1.4),
    "ne_ped_scale":   (0.6, 1.4),
    "Te_width_scale": (0.6, 1.4),
}

# Safety rail on refinement steps, in case batches stay tiny and the target is
# never reached. Not a tuning parameter -- raise it only if the run stops on
# "step limit" with the point count still climbing.
MAX_STEPS = 25

# Prefix for the retagged per-point EQDSKs: g132588_Te1.100-ne0.900-wTe1.200
EQDSK_PREFIX = "g132588"

# Where reconstruction artifacts go: the cheaseBS acceptance record, run config,
# convergence summary, baseline gfile copy and profiles files. Kept out of the
# per-point run directories so <workdir>/bXXXXpXXXX holds a GENE run and nothing
# else -- parameters, its EQDSK, its iterdb -- and can be submitted, tarred or
# handed over exactly as a hand-built run would be. None puts them in
# <workdir>/_reconstruction; point it at scratch if they need not travel with
# the runs.
RECON_DIR = None

# cheaseBS iteration limit. NOT a cost knob to trim -- the solver under-relaxes
# its bootstrap and current updates (bootstrap_mix 0.1, istar_mix 0.05), so one
# iteration applies about a twentieth of the current update and returns an
# equilibrium still carrying the BASELINE current profile no matter how far the
# input profiles were moved. The template shipped max_iter=1 until 2026-08-18,
# justified by a canary that only ever ran identity reconstructions -- where the
# loop starts at its own fixed point and has nothing to converge, so iteration
# count could not matter. Roughly 1/istar_mix = 20 iterations are needed to
# apply one full current update, more to converge it.
CHEASEBS_OVERRIDES = {
    "max_iter": 25,

    # Convergence tolerances. These, not max_iter, are what "solved to a
    # resolution limit" means -- max_iter is only the cap that stops a loop
    # that is not settling.
    #
    #   tol_bs  : bootstrap-current change between iterations
    #   tol_q   : q-profile change between iterations
    #   tol_a   : driven-current amplitude change
    #   tol_ip_rel : agreement with the TARGET Ip
    #
    # The first three measure whether the loop settled. tol_ip_rel measures
    # something else, and at its template value of 0.002 it cannot be met: the
    # reconstruction converges to a fixed point ~1.5% off target, so it vetoes
    # `converged` no matter how still the loop has gone, and every point runs to
    # max_iter. Raising it to ~0.02 (just above the measured fixed point, and
    # matching CheasebsAcceptance.max_ip_error_rel = 0.03) lets the loop stop
    # when it has actually converged.
    #
    # Left commented rather than set, because it changes what "converged" means
    # for every run and is a call to make deliberately -- ideally with the
    # advisor, and with a per-iteration trace from a PERTURBED case in hand, not
    # an identity one.
    #
    # "tol_ip_rel": 0.02,
}

In [ ]:
import os, sys, json
from datetime import datetime

sys.path.insert(0, os.path.abspath("."))
import pilot_helpers as ph

required = {"BASE_PARAMETERS": BASE_PARAMETERS}
if SEED_ITERDB:
    required.update({"SEED_ITERDB": SEED_ITERDB, "SEED_GFILE": SEED_GFILE})
else:
    required.update({"SEED_DIRPATH": SEED_DIRPATH})

print("inputs:")
missing = ph.check_inputs(required)
print()
print("imported TPED:")
tped_missing = ph.check_tped()

if BASE_PARAMETERS and os.path.exists(BASE_PARAMETERS):
    notes = ph.check_template(BASE_PARAMETERS)
    print("\ntemplate check:")
    for n in notes:
        print(f"  ! {n}")
    if not notes:
        print("  nothing to flag")

## 2. Workdir and the dummy-session warning

Everything this notebook writes goes under one timestamped directory, including
the poisoned `session.json`.

In [ ]:
WORKDIR = os.path.abspath(os.path.join(OUTROOT, datetime.now().strftime("%Y%m%d_%H-%M-%S")))
os.makedirs(WORKDIR, exist_ok=True)

with open(os.path.join(WORKDIR, "WARNING-DUMMY-SESSION.txt"), "w") as f:
    f.write("session.json here was advanced with fabricated QoI values by the\n"
            "132588 parameter-generation notebook. The sampler state is\n"
            "meaningless. Do not resume a real campaign from this directory.\n")

print(WORKDIR)

## 3. Load the seed discharge

In [ ]:
if SEED_ITERDB:
    print("seed: iterdb + gfile")
    discharge = ph.seed_from_iterdb(SEED_ITERDB, SEED_GFILE, WORKDIR)
else:
    print("seed: directory auto-discovery")
    discharge = ph.seed_from_dirpath(SEED_DIRPATH)

print("  gfile   ", discharge.gfile_filepath)
print("  pfile   ", discharge.pfile_filepath)
print("  profiles", discharge.profiles_filepaths)

## 4. Sanity-check the nominal mtanh fit

The scan axes are **scale factors on this fit**, so a bad fit does not produce a
bad point — it silently redefines what every axis value means. Check
`rms_relative` and look at the pedestal parameters before spending CHEASE-BS
runs.

In [ ]:
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh
from TPED.projects.GENE_pipelines.src.point_reconstruction import (
    PILOT_BOUNDS_132588, MTANH_AXES, MTANH_FIT_KWARGS, nominal_fits,
    is_equilibrium_axis)

if SCAN_BOUNDS is None:
    SCAN_BOUNDS = PILOT_BOUNDS_132588
    print('using TPED PILOT_BOUNDS_132588')

phys0 = DischargePhysics(discharge)

for var in sorted({v for v, _ in MTANH_AXES.values()}):
    profile, record = fit_mtanh(phys0.ds, var, **MTANH_FIT_KWARGS)
    print(f"{var}:  rms_relative = {record['rms_relative']:.4f}")
    # fit_params is MtanhProfile.as_dict() -- a dict, not a sequence.
    for k, v in record["fit_params"].items():
        print(f"    {k:<10} {v: .6g}")

print("\naxes and bounds (scale factors on the fit above):")
for name, (lo, hi) in PILOT_BOUNDS_132588.items():
    var, kwarg = MTANH_AXES[name]
    print(f"  {name:<16} {var:<3} {kwarg:<13} [{lo}, {hi}]")

Plot the seed pedestal against the fit, and against the corners of the box —
this is the cheapest way to see whether ±30% produces profiles you would
actually want CHEASE-BS to try.

In [ ]:
import matplotlib.pyplot as plt

fits = nominal_fits(phys0, set(PILOT_BOUNDS_132588))
corners = [
    ("nominal",  {}),
    ("Te low",   {"Te_ped_scale": PILOT_BOUNDS_132588["Te_ped_scale"][0]}),
    ("Te high",  {"Te_ped_scale": PILOT_BOUNDS_132588["Te_ped_scale"][1]}),
    ("wTe low",  {"Te_width_scale": PILOT_BOUNDS_132588["Te_width_scale"][0]}),
    ("wTe high", {"Te_width_scale": PILOT_BOUNDS_132588["Te_width_scale"][1]}),
]

fig, ax = plt.subplots(figsize=(7, 4))
for label, point in corners:
    from TPED.projects.GENE_pipelines.src.point_reconstruction import _apply_axes
    p = _apply_axes(phys0, point, fits=fits) if point else phys0
    ax.plot(p.rhot, p.Te, label=label, lw=1.5)
ax.set_xlim(0.8, 1.0); ax.set_xlabel(r"$\rho_{tor}$"); ax.set_ylabel("Te (eV)")
ax.legend(fontsize=8); ax.set_title("Te pedestal at the box corners")
plt.tight_layout(); plt.show()

## 5. Build the sampler and the campaign

The submitter is replaced with one that writes the parameters file and stops.
Everything before that — the sampler, `reconstruct_point`, the acceptance gate,
the ledger, `_write_parameters` — is the production path.

In [ ]:
from TPED.projects.discharge_tools.src.cheasebs_runner import CheasebsAcceptance
from TPED.projects.GENE_pipelines.src.sparse_scan_driver import SparseScanSession
from TPED.projects.GENE_pipelines.src.scan_campaign import (
    GeneScanCampaign, QoISpec, HARVESTED, PENDING, REJECTED)

# The radii the existing 132588 linear scans sit at: r_0.736 (q=4), r_0.825
# (q=5). The gate checks q here, so a point whose q has moved where the physics
# is read gets rejected rather than silently scanned.
GENE_RADII = (0.736, 0.825)

AXIS_SHORT = {"Te_ped_scale": "Te", "ne_ped_scale": "ne", "Te_width_scale": "wTe"}

sampler = SparseScanSession(SCAN_BOUNDS,
                            state_path=os.path.join(WORKDIR, "session.json"))

campaign = GeneScanCampaign(
    sampler=sampler,
    base_discharge=discharge,
    qoi=QoISpec(quantity="gamma", reduction="at_ky", ky=KY),
    workdir=WORKDIR,
    base_parameters=os.path.abspath(BASE_PARAMETERS),
    acceptance=CheasebsAcceptance.production(analysis_radii=GENE_RADII),
    ky_scanlist=[KY],
    recon_dir=RECON_DIR,
    cheasebs_overrides=CHEASEBS_OVERRIDES,
)
campaign.submitter = ph.write_parameters_only(campaign)
print("campaign ready")

## 6. Generate

Per step: propose points → reconstruct + gate each through CHEASE-BS → rename
each accepted EQDSK to carry its transform → write its parameters file → tell
the sampler fabricated values so the next step produces new points.

This is the slow cell. Each point is a CHEASE-BS run, and the loop runs until
TARGET_POINTS is reached. It always prints why it stopped.

In [ ]:
proposed = 0
step = 0
stop_reason = None

while stop_reason is None:
    if proposed >= TARGET_POINTS:
        stop_reason = f"budget reached — {proposed} point(s) >= TARGET_POINTS={TARGET_POINTS}"
        break
    if step >= MAX_STEPS:
        stop_reason = f"step limit — MAX_STEPS={MAX_STEPS} with {proposed} point(s)"
        break
    if getattr(campaign.sampler, "finished", False):
        stop_reason = f"the grid converged after {proposed} point(s)"
        break

    accepted = campaign.propose()
    batch = campaign.ledger.batch(step)
    proposed += len(batch)
    print(f"step {step}: {len(batch)} point(s), {len(accepted)} accepted, "
          f"{len(batch) - len(accepted)} rejected   [{proposed}/{TARGET_POINTS}]")
    for e in batch:
        if e.status == REJECTED:
            print(f"    {e.point_id} REJECTED — {e.note}")

    renamed = ph.retag_eqdsks(campaign, step, EQDSK_PREFIX, AXIS_SHORT)
    for pid, path in renamed.items():
        print(f"    {pid} eqdsk -> {os.path.basename(path)}")

    submitted = campaign.submit_pending()
    stuck = [e for e in campaign.ledger.batch(step) if not e.rundir and e.eqdsk]
    for e in stuck:
        print(f"    {e.point_id} PARAMETERS FAILED — {e.note}")

    # Stop before the DUMMY_QOI update below, which overwrites the ledger note
    # that submit_pending() just put the failure reason into. Advancing past a
    # failed write destroys the only diagnostic there is -- which is what
    # happened on the 22-27-39 run.
    if stuck:
        stop_reason = (f"{len(stuck)} point(s) reconstructed but produced no "
                       f"parameters file (see the notes above)")
        break
    if not batch:
        stop_reason = "the sampler proposed nothing"
        break
    if not accepted:
        stop_reason = ("nothing accepted this step — the gate is rejecting this "
                       "box, which is a bounds problem, not a code one")
        break
    if len(accepted) != len(batch):
        stop_reason = (f"batch {step} is incomplete "
                       f"({len(accepted)}/{len(batch)} accepted), so the grid "
                       f"cannot advance — tell() has no value for the missing node")
        break

    for e in campaign.ledger.batch(step):
        campaign.ledger.update(e.point_id, status=HARVESTED,
                               qoi=ph.dummy_qoi(e.point), note="DUMMY_QOI")
    campaign.tell_batch(step)
    print(f"    advanced the grid on {len(submitted)} fabricated value(s)")
    step += 1

# Always say why it stopped. Without this a break at step 0 and a completed
# scan print the same thing, which is how a one-point run looked like a
# finished one.
print(f"\nSTOPPED: {stop_reason}")
print(f"{proposed} point(s) proposed over {step + 1} step(s)")

## 7. Did it work?

Reads every written parameters file back. The three silent failure modes: a scan
axis written into the namelist as a GENE key, an `iterdb_file` still pointing at
the seed, and a `geomfile` that is not this point's EQDSK.

In [ ]:
problems = ph.verify_parameters(campaign, is_equilibrium_axis)

for e in campaign.ledger.entries.values():
    if not e.rundir:
        continue
    print(f"{e.point_id}  {json.dumps(e.point, sort_keys=True)}")
    print(f"    eqdsk   {e.eqdsk}")
    print(f"    iterdb  {e.iterdb}")
    print(f"    params  {os.path.join(e.rundir, 'parameters')}")

if problems:
    print(f"\n{len(problems)} PROBLEM(S):")
    for pid, msg in problems:
        print(f"  {pid}: {msg}")
else:
    print("\nno problems found in the written parameters files")

Print one parameters file in full — the automated checks only catch what they
were told to look for.

In [ ]:
written = [e for e in campaign.ledger.entries.values() if e.rundir]
if written:
    print(open(os.path.join(written[0].rundir, "parameters")).read())
else:
    print("nothing was written")

## 8. Every generated profile, side by side

Reads each point's `profiles_e` / `profiles_i` back off disk — what CHEASE-BS
was actually handed, not a recomputation of the transform. The check that
matters is the last one: if two points share a profile, the transform did not
take, and every CHEASE-BS run would still have succeeded while the axes scanned
nothing.

In [ ]:
profile_rows = ph.summarize_profiles(campaign, AXIS_SHORT)

In [ ]:
ph.plot_profiles(profile_rows)

The same points through discharge_tools' own plotting, rebuilt from each
point's written profiles and its own reconstructed EQDSK. `full=True` swaps the
profiles-only view for the report layout, which adds the flux-surface geometry —
the only view here that shows whether the *equilibria* differ rather than just
the profiles that produced them.

In [ ]:
fig = ph.plot_discharge_overlay(campaign, AXIS_SHORT, rhot=[0.85, 1.0])

In [ ]:
# Slower: reads every point's EQDSK and draws its flux surfaces.
fig = ph.plot_discharge_overlay(campaign, AXIS_SHORT, full=True)

### Did cheaseBS actually converge?

Read the iteration count before reading anything else on this page. An
equilibrium that stopped at iteration 1 is not converged — it is the baseline
current profile with different input profiles underneath, because the solver
under-relaxes and one iteration applies a fraction of the update. If every point
stops at exactly `CHEASEBS_OVERRIDES['max_iter']`, the tolerances were never met and the
error below is what the solver *achieved*, not what it promised.

In [ ]:
conv_rows = ph.convergence_report(campaign)

## 9. Do the equilibria differ?

Flux-surface contours are the wrong instrument for this question and will
mislead. The reconstruction preserves the source EFIT boundary by design, and a
pedestal perturbation is a small fraction of stored energy — so the separatrix
*cannot* move, and interior surfaces shift by far less than a contour plot
resolves. Identical-looking flux surfaces are the expected result, not evidence
that nothing happened.

What must move is the flux functions. The table below gives the max relative
difference of F, p, p', FF', q and the 2-D psi map against the reference point,
and the normalized flux coordinate where that max sits — which is the part
overlapping curves cannot tell you. It also checks the boundary really did stay
fixed, since a boundary that moved would be a finding.

In [ ]:
gfile_rows = ph.compare_gfiles(campaign, AXIS_SHORT)

Differences, not overlays — a sub-percent change between two curves drawn on
the same axes is one curve.

In [ ]:
ph.plot_gfile_differences(gfile_rows)

## 10. Summary record

In [ ]:
summary = os.path.join(WORKDIR, "pilot_summary.json")
with open(summary, "w") as f:
    json.dump({
        "workdir": WORKDIR,
        "base_parameters": os.path.abspath(BASE_PARAMETERS),
        "seed": {"dirpath": SEED_DIRPATH, "iterdb": SEED_ITERDB,
                 "gfile": SEED_GFILE},
        "axes": SCAN_BOUNDS,
        "ky": KY,
        "analysis_radii": list(GENE_RADII),
        "qoi_values_are_fabricated": True,
        "profiles": [{k: v for k, v in r.items()
                      if k not in ("rhot", "Te", "ne", "Ti")}
                     for r in profile_rows],
        "problems": problems,
        "points": [{"point_id": e.point_id, "batch": e.batch, "point": e.point,
                    "status": e.status, "eqdsk": e.eqdsk, "iterdb": e.iterdb,
                    "rundir": e.rundir, "note": e.note}
                   for e in campaign.ledger.entries.values()],
    }, f, indent=1)
print(summary)